In [1]:
# Imports
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50
from sklearn.neighbors import NearestNeighbors
from PIL import Image, ImageDraw, ImageFont

In [2]:
# Function
def preprocess_frame(frame):
        preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
                             ])
        return preprocess(frame)

In [3]:
# Model
num_classes = 1081
resnet_checkpoint_path = 'models/resnet50_weights_best_acc.tar'

model = resnet50(num_classes=num_classes)
checkpoint = torch.load(resnet_checkpoint_path, map_location='cpu')
state_dict = checkpoint["model"]

model = resnet50(num_classes=num_classes)
model.load_state_dict(state_dict=state_dict, strict=True)
model = torch.nn.Sequential(*(list(model.children())[:-1]))
model.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [4]:
# Datasets
dataset_1 = "data/frames/biodiverse_1/"
frames_1 = os.listdir(dataset_1)

dataset_2 = "data/frames/biodiverse_2/"
frames_2 = os.listdir(dataset_2)

In [5]:
# Embeddings
embeddings1 = []
for frame in frames_1:
    img = Image.open(dataset_1 + frame)
    preprocessed_img = preprocess_frame(img)
    input_tensor = preprocessed_img.unsqueeze(0)
    with torch.no_grad():
        features = model(input_tensor)
    embeddings1.append(features.cpu().numpy().flatten())
embeddingspace1 = np.array(embeddings1)

embeddings2 = []
for frame in frames_2:
    img = Image.open(dataset_2 + frame)
    preprocessed_img = preprocess_frame(img)
    input_tensor = preprocessed_img.unsqueeze(0)
    with torch.no_grad():
        features = model(input_tensor)
    embeddings2.append(features.cpu().numpy().flatten())
embeddingspace2 = np.array(embeddings2)

In [6]:
# Density calculation
k = 10

nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto', metric='cosine').fit(embeddingspace1)
distances, indices = nbrs.kneighbors(embeddingspace1)
distances = distances[:, 1:] # remove distance to self
density = 1 / (np.mean(distances, axis=1) + 1e-10)
average_density1 = np.mean(density)

print("Average kNN (k = 10) density of initial embedding space: ", average_density1)

Average kNN (k = 10) density of initial embedding space:  7.7963843


In [7]:
# Algorithm
output_path = "data/frames/output_labeled"
fontsize = 50

threshold = average_density1

for frame in frames_2:
        img = Image.open(dataset_2 + frame)
        preprocessed_img = preprocess_frame(img)
        input_tensor = preprocessed_img.unsqueeze(0)
        with torch.no_grad():
            features = model(input_tensor)
        embedding = features.cpu().numpy().flatten()

        embeddings1.append(embedding)
        embeddingspace = np.array(embeddings1)
        new_point = embeddingspace[-1]

        nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto', metric='cosine').fit(embeddingspace)
        distances, indices = nbrs.kneighbors([new_point])
        distances = distances[:, 1:] # Remove distance to self
        indices = indices[:, 1:]
        avg_distance = np.mean(distances)
        point_density = 1 / (avg_distance + 1e-10)
    
        if (point_density >= threshold):
            closest_idx = indices[0, 0]
            del embeddings1[closest_idx]
            del embeddings1[-1]
            label = "mow"
        else:
            label = "don't mow"
        
        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.truetype("arial.ttf", size=fontsize)
        except:
            font = ImageFont.load_default()
        width = draw.textlength(label, font=font)
        height = fontsize
        x, y = 10, 10
        color = (0, 200, 0) if label == "mow" else (200, 0, 0)
        draw.rectangle([x, y, x + width + 10, y + height + 5], fill=(255, 255, 255, 128))
        draw.text((x+5, y+5), label, fill=color, font=font)

        save_path = os.path.join(output_path, frame)
        img.save(save_path)

        print(frame + " saved")

frame_b2_0000.jpg saved
frame_b2_0321.jpg saved
frame_b2_0642.jpg saved
frame_b2_0963.jpg saved
frame_b2_10272.jpg saved
frame_b2_10593.jpg saved
frame_b2_10914.jpg saved
frame_b2_11235.jpg saved
frame_b2_11556.jpg saved
frame_b2_11877.jpg saved
frame_b2_12198.jpg saved
frame_b2_12519.jpg saved
frame_b2_1284.jpg saved
frame_b2_12840.jpg saved
frame_b2_13161.jpg saved
frame_b2_13482.jpg saved
frame_b2_13803.jpg saved
frame_b2_14124.jpg saved
frame_b2_14445.jpg saved
frame_b2_14766.jpg saved
frame_b2_15087.jpg saved
frame_b2_15408.jpg saved
frame_b2_15729.jpg saved
frame_b2_1605.jpg saved
frame_b2_16050.jpg saved
frame_b2_16371.jpg saved
frame_b2_16692.jpg saved
frame_b2_17013.jpg saved
frame_b2_17334.jpg saved
frame_b2_17655.jpg saved
frame_b2_17976.jpg saved
frame_b2_18297.jpg saved
frame_b2_18618.jpg saved
frame_b2_18939.jpg saved
frame_b2_1926.jpg saved
frame_b2_19260.jpg saved
frame_b2_19581.jpg saved
frame_b2_19902.jpg saved
frame_b2_20223.jpg saved
frame_b2_20544.jpg saved
frame_b

In [8]:
# Interpolate all frames
output_path = "data/frames/output_labeled"
fontsize = 50

threshold = average_density1

frame_idxs = []
frame_lbls = {}

for frame in frames_2:
        idx = frame.split('_')
        idx = idx[2].split('.')
        idx = idx[0]
        frame_idxs.append(int(idx))

        img = Image.open(dataset_2 + frame)
        preprocessed_img = preprocess_frame(img)
        input_tensor = preprocessed_img.unsqueeze(0)
        with torch.no_grad():
            features = model(input_tensor)
        embedding = features.cpu().numpy().flatten()

        embeddings1.append(embedding)
        embeddingspace = np.array(embeddings1)
        new_point = embeddingspace[-1]

        nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto', metric='cosine').fit(embeddingspace)
        distances, indices = nbrs.kneighbors([new_point])
        distances = distances[:, 1:] # Remove distance to self
        indices = indices[:, 1:]
        avg_distance = np.mean(distances)
        point_density = 1 / (avg_distance + 1e-10)
    
        if (point_density >= threshold):
            closest_idx = indices[0, 0]
            del embeddings1[closest_idx]
            del embeddings1[-1]
            label = "mow"
        else:
            label = "don't mow"
        
        frame_lbls[int(idx)] = label

interpolated_lbls = []

for i in range(30243):

    int_keys = [int(f) for f in frame_lbls.keys()]

    before = max([f for f in int_keys if f <= i], default=None)
    after = min([f for f in int_keys if f >= i], default=None)

    if before is None:
        interpolated_lbl = frame_lbls[after]
    elif after is None:
        interpolated_lbl = frame_lbls[before]
    else:
        interpolated_lbl = frame_lbls[before] if (i - before) < (after - i) else frame_lbls[after]
    
    interpolated_lbls.append(interpolated_lbl)

In [9]:
# Annotate video
cap = cv2.VideoCapture("data/videos/biodiverse.mp4")
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter("data/videos/annotated.mp4", cv2.VideoWriter_fourcc(*'mp4v'), 60, (width, height))

i = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    label = interpolated_lbls[i]
    color = (0, 200, 0) if label == "mow" else (200, 0, 0)

    cv2.putText(frame, label, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, color, 3)

    out.write(frame)
    i += 1

cap.release()
out.release()